# Media-bias classifier: DeBERTa-v3 / XLM-R on BABE

**Kaggle settings (right sidebar):** Accelerator → **GPU T4 x2** (or P100) · Internet → **On** · Persistence → Files (optional).
Optional: add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) to push models to Hugging Face or use gated datasets.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
# one GPU is enough; avoids multi-GPU DataParallel surprises
!nvidia-smi

In [ ]:
!git clone -q https://github.com/Sidvortex/media-bias-finetune.git || (cd media-bias-finetune && git pull -q)
%cd media-bias-finetune
!pip install -q -U -r requirements.txt

In [ ]:
# optional: log in to Hugging Face with a Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception as e:
    print('No HF_TOKEN secret, continuing anonymously:', type(e).__name__)

## 1. DeBERTa-v3-base on BABE (≈10 min)

In [ ]:
!python train.py --config configs/deberta_babe.yaml

## 2. Multilingual XLM-R (for later Indian-news fine-tuning)

In [ ]:
!python train.py --config configs/xlmr_babe.yaml

## 3. Compare

In [ ]:
import json
for run in ['deberta-babe', 'xlmr-babe']:
    r = json.load(open(f'outputs/{run}/results.json'))
    print(f"{run:14s} acc={r['test_accuracy']:.4f}  macro-F1={r['test_macro_f1']:.4f}")

## 4. Score an article

In [ ]:
open('article.txt','w').write('''The government quietly pushed through its controversial bill late on Friday. Officials said the change would take effect next month. Critics slammed the reckless move as a blatant power grab.''')
!python predict.py --model outputs/deberta-babe/best --file article.txt

## 5. Save the models

In [ ]:
!cd outputs && zip -qr /kaggle/working/bias_models.zip deberta-babe/best deberta-babe/results.json xlmr-babe/best xlmr-babe/results.json
!ls -lh /kaggle/working/*.zip